# SVG Scaling Laws — Large Model Training

In [ ]:
# ── 1. Mount Google Drive ──────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Verify GPU & install nothing (torch is pre-installed in Colab) ──────
import torch
assert torch.cuda.is_available(), 'No GPU found — check runtime type!'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU : {gpu}')
print(f'VRAM: {vram:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'bfloat16 supported: {torch.cuda.is_bf16_supported()}')

In [ ]:
# ── 3. Configure paths — edit DRIVE_ROOT if you used a different folder ───
import os

DRIVE_ROOT  = '/content/drive/MyDrive/ML-Final-Project'
TRAIN_NPY   = f'{DRIVE_ROOT}/data/tokenized/train.npy'
VAL_NPY     = f'{DRIVE_ROOT}/data/tokenized/val.npy'
OUTPUT_DIR  = f'{DRIVE_ROOT}/outputs'   # checkpoints + logs saved here

os.makedirs(f'{OUTPUT_DIR}/checkpoints/large', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/logs',              exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/results',           exist_ok=True)

assert os.path.exists(TRAIN_NPY), f'train.npy not found at {TRAIN_NPY}'
assert os.path.exists(VAL_NPY),   f'val.npy not found at {VAL_NPY}'
print('Data files found.')
print(f'  train.npy: {os.path.getsize(TRAIN_NPY)/1e6:.1f} MB')
print(f'  val.npy  : {os.path.getsize(VAL_NPY)/1e6:.1f} MB')

In [ ]:
# ── 4. Model definition (GPT — identical to scripts/model.py) ─────────────
import math
from dataclasses import dataclass
import torch.nn as nn
from torch.nn import functional as F

@dataclass
class ModelConfig:
    vocab_size: int = 1024
    block_size: int = 1024
    d_model:    int = 512
    n_layers:   int = 10
    n_heads:    int = 8
    d_ff:       int = 2048
    dropout:  float = 0.1
    bias:      bool = False


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_heads  = cfg.n_heads
        self.d_model  = cfg.d_model
        self.head_dim = cfg.d_model // cfg.n_heads
        self.drop_p   = cfg.dropout
        self.c_attn   = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.c_proj   = nn.Linear(cfg.d_model, cfg.d_model,     bias=cfg.bias)
        self.resid_drop = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.d_model, dim=2)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        y = F.scaled_dot_product_attention(
            q, k, v, dropout_p=self.drop_p if self.training else 0.0, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.c_proj(y))


class MLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc   = nn.Linear(cfg.d_model, cfg.d_ff,    bias=cfg.bias)
        self.proj = nn.Linear(cfg.d_ff,    cfg.d_model, bias=cfg.bias)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.proj(self.act(self.fc(x))))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1  = nn.LayerNorm(cfg.d_model, bias=cfg.bias)
        self.attn = CausalSelfAttention(cfg)
        self.ln2  = nn.LayerNorm(cfg.d_model, bias=cfg.bias)
        self.mlp  = MLP(cfg)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.transformer = nn.ModuleDict(dict(
            wte  = nn.Embedding(cfg.vocab_size, cfg.d_model),
            wpe  = nn.Embedding(cfg.block_size, cfg.d_model),
            drop = nn.Dropout(cfg.dropout),
            h    = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)]),
            ln_f = nn.LayerNorm(cfg.d_model, bias=cfg.bias),
        ))
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith('proj.weight'):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.n_layers))

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device)
        x    = self.transformer.drop(
                   self.transformer.wte(idx) + self.transformer.wpe(pos))
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss   = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                     targets.view(-1), ignore_index=-1)
            return logits, loss
        return self.lm_head(x[:, [-1], :]), None

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())


cfg   = ModelConfig()
model = GPT(cfg)
print(f'Large model parameters: {model.count_parameters():,}')

In [ ]:
# ── 5. Training hyperparameters ────────────────────────────────────────────
import numpy as np

# -- identical settings to local training --
LR                    = 3e-3
EFFECTIVE_BATCH_TOKENS = 65536
BATCH_SIZE            = 8      # larger physical batch — A100 has plenty of memory
BLOCK_SIZE            = 1024
WEIGHT_DECAY          = 0.1
WARMUP_FRACTION       = 0.05
GRAD_CLIP             = 1.0
EVAL_INTERVAL         = 100
SEED                  = 42
USE_BF16              = torch.cuda.is_bf16_supported()   # True on A100

GRAD_ACCUM = max(1, EFFECTIVE_BATCH_TOKENS // (BATCH_SIZE * BLOCK_SIZE))
TOKENS_PER_STEP = BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda')

print(f'Precision : {"bfloat16" if USE_BF16 else "float32"}')
print(f'Phys batch: {BATCH_SIZE} seqs  |  Grad accum: {GRAD_ACCUM}')
print(f'Eff. batch: {TOKENS_PER_STEP:,} tokens/step')

In [ ]:
# ── 6. Load data ───────────────────────────────────────────────────────────
print('Loading data into RAM...')
train_data = np.array(np.load(TRAIN_NPY, mmap_mode='r'))
val_data   = np.array(np.load(VAL_NPY,   mmap_mode='r'))
print(f'Train: {len(train_data):,} tokens  |  Val: {len(val_data):,} tokens')
print(f'Token range — train: [0, {int(train_data.max())}]  val: [0, {int(val_data.max())}]')

def get_batch(data, bs):
    ix = np.random.randint(0, len(data) - BLOCK_SIZE, size=(bs,))
    x  = torch.stack([torch.from_numpy(data[i:i+BLOCK_SIZE].astype(np.int64)) for i in ix])
    y  = torch.stack([torch.from_numpy(data[i+1:i+BLOCK_SIZE+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def full_val_loss():
    model.eval()
    losses, bs = [], 8
    for i in range(0, len(val_data) - BLOCK_SIZE, BLOCK_SIZE * bs):
        starts = list(range(i, min(i + BLOCK_SIZE*bs, len(val_data)-BLOCK_SIZE), BLOCK_SIZE))
        if not starts: break
        x = torch.stack([torch.from_numpy(val_data[s:s+BLOCK_SIZE].astype(np.int64)) for s in starts]).to(device)
        y = torch.stack([torch.from_numpy(val_data[s+1:s+BLOCK_SIZE+1].astype(np.int64)) for s in starts]).to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))

@torch.no_grad()
def quick_val_loss(n_batches=50):
    # Stay in train mode — toggling eval with torch.compile triggers CUDA assert.
    # full_val_loss() (called once at end) is the only place that uses eval mode.
    losses = []
    for _ in range(n_batches):
        x, y = get_batch(val_data, BATCH_SIZE)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            _, loss = model(x, y)
        losses.append(loss.item())
    return float(np.mean(losses))

In [ ]:
# ── 7. Build model + optimizer ─────────────────────────────────────────────
model = model.to(device)

# torch.compile disabled — it triggers device-side asserts on some CUDA configs.
# The A100 reaches 140-250k tok/s uncompiled; total run time is still ~15 min.
print('torch.compile: disabled (stability)')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR,
    betas=(0.9, 0.95), eps=1e-8, weight_decay=WEIGHT_DECAY
)

total_steps   = math.ceil(len(train_data) / TOKENS_PER_STEP)
warmup_steps  = max(1, int(total_steps * WARMUP_FRACTION))
min_lr        = LR * 0.1

def get_lr(step):
    if step < warmup_steps:
        return LR * step / max(warmup_steps, 1)
    if step >= total_steps:
        return min_lr
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return min_lr + 0.5 * (1 + math.cos(math.pi * progress)) * (LR - min_lr)

print(f'Total steps : {total_steps:,}')
print(f'Warmup steps: {warmup_steps}')
print(f'Optimizer   : AdamW  lr={LR:.1e}  wd={WEIGHT_DECAY}')

In [ ]:
# ── 8. Training loop ───────────────────────────────────────────────────────
import time, json

scaler       = torch.amp.GradScaler('cuda', enabled=False)  # bf16 has same dynamic range as fp32 — no loss scaling needed
train_losses = []
tokens_seen  = 0
peak_mem_mb  = 0.0
t_start      = time.time()

model.train()
for step in range(total_steps):
    current_lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = current_lr

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        x, y = get_batch(train_data, BATCH_SIZE)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            _, loss = model(x, y)
        loss = loss / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()

    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    tokens_seen += TOKENS_PER_STEP
    mem = torch.cuda.max_memory_allocated() / 1024**2
    if mem > peak_mem_mb:
        peak_mem_mb = mem

    if step % EVAL_INTERVAL == 0 or step == total_steps - 1:
        elapsed = time.time() - t_start
        tps     = tokens_seen / elapsed
        v_loss  = quick_val_loss(50)
        pct     = 100 * step / total_steps
        remaining = (total_steps - step) * (elapsed / max(step, 1))
        train_losses.append({'step': step, 'train_loss': round(step_loss, 5),
                              'val_loss': round(v_loss, 5), 'lr': round(current_lr, 8)})
        print(f'  step {step:5d}/{total_steps} ({pct:4.1f}%)  '
              f'train={step_loss:.4f}  val={v_loss:.4f}  '
              f'lr={current_lr:.2e}  {tps:,.0f} tok/s  '
              f'ETA={remaining/60:.0f}min  mem={peak_mem_mb:.0f}MB')

print('\nComputing final validation loss...')
final_val = full_val_loss()
total_time = time.time() - t_start
avg_tps    = tokens_seen / total_time

print(f'Final val loss : {final_val:.4f}')
print(f'Total time     : {total_time/3600:.2f} hrs')
print(f'Avg throughput : {avg_tps:,.0f} tok/s')
print(f'Peak GPU memory: {peak_mem_mb:.1f} MB')

In [ ]:
# ── 9. Save checkpoint + results to Drive ─────────────────────────────────
# Save model checkpoint
ckpt_path = f'{OUTPUT_DIR}/checkpoints/large/checkpoint_final.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_loss': final_val,
    'config': {'name':'large','d_model':512,'n_layers':10,'n_heads':8,'d_ff':2048,
               'vocab_size':1024,'block_size':1024,'dropout':0.1,'bias':False},
    'lr': LR,
    'step': total_steps,
}, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')

# Save training log
result = {
    'name': 'large',
    'param_count': 32516608,
    'val_loss': final_val,
    'lr': LR,
    'd_model': 512, 'n_layers': 10, 'n_heads': 8, 'd_ff': 2048,
    'train_losses': train_losses,
    'total_time_seconds': round(total_time, 1),
    'tokens_per_second': round(avg_tps, 1),
    'peak_memory_mb': round(peak_mem_mb, 1),
    'total_steps': total_steps,
    'effective_batch_tokens': TOKENS_PER_STEP,
}
log_path = f'{OUTPUT_DIR}/logs/large_lr3e-3.json'
with open(log_path, 'w') as f:
    json.dump(result, f, indent=2)
print(f'Log saved: {log_path}')

# Append to scaling_results.json (to merge back with local results later)
scaling_path = f'{OUTPUT_DIR}/results/scaling_results.json'
entry = {k: v for k, v in result.items() if k != 'train_losses'}
existing = []
if os.path.exists(scaling_path):
    with open(scaling_path) as f:
        existing = json.load(f)
existing = [r for r in existing if r.get('name') != 'large']
existing.append(entry)
existing.sort(key=lambda r: r.get('param_count', 0))
with open(scaling_path, 'w') as f:
    json.dump(existing, f, indent=2)
print(f'Scaling results saved: {scaling_path}')

In [ ]:
# ── 10. Plot training curve ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

steps  = [e['step']       for e in train_losses]
t_loss = [e['train_loss'] for e in train_losses]
v_loss = [e['val_loss']   for e in train_losses]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(steps, t_loss, color='steelblue', linewidth=1.8)
axes[0].set_title('Train Loss — Large Model', fontsize=12)
axes[1].plot(steps, v_loss, color='tomato',    linewidth=1.8)
axes[1].set_title('Val Loss — Large Model', fontsize=12)
for ax in axes:
    ax.set_xlabel('Step'); ax.set_ylabel('Loss'); ax.grid(alpha=0.3)
fig.suptitle(f'Large (~32.5M params) | LR=3e-3 | Final val={final_val:.4f}', fontsize=11)
fig.tight_layout()

plot_path = f'{OUTPUT_DIR}/plots/large_training_curve.png'
os.makedirs(os.path.dirname(plot_path), exist_ok=True)
fig.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved: {plot_path}')

In [ ]:
# ── 11. Summary ────────────────────────────────────────────────────────────
print('=' * 55)
print('LARGE MODEL TRAINING COMPLETE')
print('=' * 55)
print(f'  Final val loss  : {final_val:.4f}')
print(f'  Total time      : {total_time/3600:.2f} hrs')
print(f'  Avg throughput  : {avg_tps:,.0f} tok/s')
print(f'  Peak GPU memory : {peak_mem_mb:.0f} MB')
print()
print('Files saved to Google Drive:')
print(f'  Checkpoint : {ckpt_path}')
print(f'  Log        : {log_path}')
print(f'  Results    : {scaling_path}')
print()
print('Next step: download these files to your Mac and place them at:')
print('  outputs/checkpoints/large/checkpoint_final.pt')
print('  outputs/logs/large_lr3e-3.json')
print('Then merge large entry into your local scaling_results.json')
print('and run: python3 scripts/fit_scaling_law.py')